# Course 2 — Logistic Regression: Complete

From the fundamental problem of applying linear regression to classification all the way through
multinomial softmax, ROC curves, regularization, and the limits of linear classifiers.

**Sections**
1. Why not linear regression for classification? (0:00–0:20)
2. Binary logistic regression — model, MLE, coefficients, confounding (0:20–0:50)
3. Multiple logistic regression on Smarket (0:50–1:10)
4. Case-control sampling and the intercept correction (1:10–1:25)
5. Multinomial logistic regression — softmax (1:25–1:50)
6. ROC curves, AUC, and threshold tuning (1:50–2:15)
7. L1 and L2 regularization (2:15–2:40)
8. When linear fails — motivating non-linear classifiers (2:40–3:00)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_moons
from sklearn.metrics import (classification_report, roc_curve, auc,
                              confusion_matrix, f1_score, roc_auc_score)

default = load_dataset('default')
smarket = load_dataset('smarket')
iris    = load_dataset('iris')
d = default.copy()
d['y'] = (d['default'] == 'Yes').astype(int)
print('default:', d.shape, '  smarket:', smarket.shape, '  iris:', iris.shape)

## 1. Why not linear regression for classification?

The naive approach: encode Y = 1 (default) / 0 (no default) and fit OLS.
The problem: predictions escape [0, 1] — they are not valid probabilities.

In [ ]:
from sklearn.linear_model import LinearRegression

X_bal = d[['balance']].to_numpy()
y_bin = d['y'].to_numpy()

lin = LinearRegression().fit(X_bal, y_bin)
log = LogisticRegression(C=1e6, max_iter=2000).fit(X_bal, y_bin)

xs = np.linspace(0, 2700, 300).reshape(-1, 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, model, title, preds in zip(
        axes,
        [lin, log],
        ['Linear regression (OLS)', 'Logistic regression'],
        [lin.predict(xs), log.predict_proba(xs)[:, 1]]):
    ax.scatter(X_bal.ravel(), y_bin, s=4, alpha=0.15, color='steelblue')
    ax.plot(xs, preds, color='C3', linewidth=2)
    ax.axhline(0, color='k', linestyle='--', linewidth=0.7, alpha=0.5)
    ax.axhline(1, color='k', linestyle='--', linewidth=0.7, alpha=0.5)
    ax.set_xlabel('balance'); ax.set_ylabel('P(default)')
    ax.set_title(title)
plt.tight_layout(); plt.show()

print(f'Linear prediction at balance=0:    {lin.predict([[0]])[0]:.3f}  (negative!)')
print(f'Linear prediction at balance=2700: {lin.predict([[2700]])[0]:.3f}  (above 1!)')
print(f'Logistic at balance=0:    {log.predict_proba([[0]])[0, 1]:.6f}')
print(f'Logistic at balance=2700: {log.predict_proba([[2700]])[0, 1]:.6f}')

## 2. Binary Logistic Regression

### The model

$$P(y=1 \mid x) = \sigma(\beta_0 + \beta_1 x) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x)}}$$

Equivalently in log-odds (logit) form:
$$\log\frac{p}{1-p} = \beta_0 + \beta_1 x$$

**Maximum likelihood estimation**: maximize the log-likelihood
$$\ell(\beta) = \sum_i \left[ y_i \log p(x_i) + (1-y_i) \log(1-p(x_i)) \right]$$

This function is **concave** in β — any optimizer finds the unique global maximum.

**Odds ratio**: $e^{\beta_j}$ is the multiplicative change in odds per unit increase in $x_j$.

In [ ]:
z = np.linspace(-6, 6, 300)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(z, 1/(1+np.exp(-z)), linewidth=2, color='C0')
ax.axhline(0.5, color='k', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_xlabel(r'$\beta_0 + \beta_1 x$ (log-odds)')
ax.set_ylabel('p(x)'); ax.set_title('The sigmoid function')
ax.set_ylim(-0.05, 1.05); plt.show()

In [ ]:
m1 = LogisticRegression(C=1e6, max_iter=2000).fit(X_bal, y_bin)
print(f'Intercept (β₀) = {m1.intercept_[0]:.4f}')
print(f'β_balance      = {m1.coef_[0][0]:.6f}')
print(f'Odds ratio per +$1  balance: {np.exp(m1.coef_[0][0]):.6f}')
print(f'Odds ratio per +$100 balance: {np.exp(100*m1.coef_[0][0]):.4f}  (×1.73)')

### Coefficient table (ISLR Table 4.1 style)

Using `statsmodels` to get standard errors and z-statistics:

In [ ]:
try:
    import statsmodels.api as sm
    X_sm = sm.add_constant(X_bal)
    glm = sm.GLM(y_bin, X_sm, family=sm.families.Binomial()).fit()
    print(glm.summary2().tables[1].round(4))
except ImportError:
    print('statsmodels not available — showing sklearn coefficients only')
    print(f'β₀={m1.intercept_[0]:.4f}, β_balance={m1.coef_[0][0]:.6f}')

### Confounding: the student paradox

**Simple regression** (student only): β_student > 0 → students default *more*.  
**Multiple regression** (student + balance): β_student < 0 → controlling for balance, students default *less*.

Why? Students carry higher balances on average. Once balance is controlled, student status is protective.
This is **confounding** — marginal and conditional associations can have opposite signs.

In [ ]:
d['student_num'] = (d['student'] == 'Yes').astype(float)

# Simple: student only
m_simple = LogisticRegression(C=1e6, max_iter=2000).fit(d[['student_num']], y_bin)
print(f'Simple regression — β_student = {m_simple.coef_[0][0]:.4f}  (positive: students default more)')

# Multiple: student + balance + income
X_multi = d[['balance', 'income', 'student_num']]
m_multi = LogisticRegression(C=1e6, max_iter=2000).fit(X_multi, y_bin)
coefs = pd.Series(m_multi.coef_[0], index=X_multi.columns)
print('\nMultiple regression:')
print(coefs.round(6).to_string())
print('\nβ_student < 0: controlling for balance, students default LESS (confounding!)')

# Visualize: default rate by balance, colored by student status
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
xs = np.linspace(0, 2700, 300).reshape(-1, 1)
for is_student, label, color in [(0, 'Non-student', 'C0'), (1, 'Student', 'C3')]:
    X_pred = np.column_stack([xs, np.full_like(xs, d['income'].mean()), np.full_like(xs, is_student)])
    ps = m_multi.predict_proba(X_pred)[:, 1]
    axes[0].plot(xs, ps, label=label, color=color, linewidth=2)
    subset = d[d['student_num'] == is_student]
    axes[1].scatter(subset['balance'], subset['y'], s=3, alpha=0.2, color=color, label=label)
axes[0].set_xlabel('balance'); axes[0].set_ylabel('P(default)')
axes[0].set_title('Default probability by balance (controlling for income)')
axes[0].legend()
axes[1].set_xlabel('balance'); axes[1].set_ylabel('default (0/1)')
axes[1].set_title('Raw data by student status')
axes[1].legend()
plt.tight_layout(); plt.show()

## 3. Multiple Logistic Regression on Smarket

Predict market direction (Up/Down) from lag returns.
**Never** include `Today` as a predictor — it directly encodes the direction we are predicting.

In [ ]:
Xs = smarket[['Lag1', 'Lag2', 'Lag3', 'Lag4', 'Lag5', 'Volume']]
ys = (smarket['Direction'] == 'Up').astype(int)

m_smarket = LogisticRegression(C=1e6, max_iter=2000).fit(Xs, ys)
print('Coefficients:')
print(pd.Series(m_smarket.coef_[0], index=Xs.columns).round(4).to_string())
print(f'\nTraining accuracy = {m_smarket.score(Xs, ys):.4f}')

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(Xs, ys, test_size=0.3, random_state=0, stratify=ys)
m_s2 = LogisticRegression(C=1e6, max_iter=2000).fit(Xtr, ytr)
print(f'Held-out accuracy = {m_s2.score(Xte, yte):.4f}  (barely above chance)')
print('Lesson: a valid model on clean data can still find no signal. Markets are hard.')

## 4. Case-Control Sampling and the Intercept Correction

In medical studies, disease events (cases) are rare. Researchers deliberately **oversample cases**
and undersample controls to reduce data collection costs.

This biases the **intercept** β̂₀. The fix:

$$\hat{\beta}_0^{\text{corrected}} = \hat{\beta}_0 + \log\frac{\tilde{\pi}}{1-\tilde{\pi}} - \log\frac{\pi}{1-\pi}$$

where:
- π = true population prevalence (e.g., 0.01 for rare disease)
- π̃ = fraction of cases in the study (e.g., 0.50)

**All other coefficients β̂₁, …, β̂ₚ are unbiased** under case-control sampling. Only the intercept needs correction.
This makes logistic regression the gold standard for epidemiological case-control studies.

In [ ]:
# Demonstrate the intercept correction on the Default dataset
# True prevalence in Default: ~3.3%
pi_true = d['y'].mean()      # true population rate
pi_tilde = 0.5               # hypothetical case-control study: 50% cases

# Fit on full data (unbiased reference)
X_bal_inc = d[['balance', 'income']].to_numpy()
m_full = LogisticRegression(C=1e6, max_iter=2000).fit(X_bal_inc, y_bin)

# Simulate a biased case-control sample
idx_pos = np.where(y_bin == 1)[0]
idx_neg = np.where(y_bin == 0)[0]
rng = np.random.default_rng(42)
n_cases = len(idx_pos)
idx_neg_sample = rng.choice(idx_neg, size=n_cases, replace=False)  # equal case/control
idx_cc = np.concatenate([idx_pos, idx_neg_sample])
X_cc, y_cc = X_bal_inc[idx_cc], y_bin[idx_cc]

m_cc = LogisticRegression(C=1e6, max_iter=2000).fit(X_cc, y_cc)

# Apply intercept correction
correction = np.log(pi_tilde/(1-pi_tilde)) - np.log(pi_true/(1-pi_true))
b0_corrected = m_cc.intercept_[0] - correction

print(f'True prevalence π = {pi_true:.4f}, study rate π̃ = {pi_tilde:.4f}')
print(f'\nFull-data fit:       β₀ = {m_full.intercept_[0]:.4f}, β_balance = {m_full.coef_[0][0]:.6f}')
print(f'Case-control fit:    β₀ = {m_cc.intercept_[0]:.4f}, β_balance = {m_cc.coef_[0][0]:.6f}')
print(f'Corrected intercept: β₀ = {b0_corrected:.4f}')
print(f'\nSlopes are the same: β_balance unchanged = {m_cc.coef_[0][0]:.6f}')

## 5. Multinomial Logistic Regression — Softmax

When Y has K > 2 classes, the sigmoid generalizes to the **softmax** function:

$$P(Y=k \mid X=x) = \frac{e^{\beta_k^\top x}}{\sum_{j=1}^{K} e^{\beta_j^\top x}}$$

- K weight vectors (rows of `coef_` in sklearn), but only K−1 are identifiable (one class is the reference).
- Outputs always sum to 1 — a valid probability distribution.
- For K=2, softmax reduces exactly to the binary sigmoid.

### Decision boundary geometry

Between any two classes k and l, the boundary is where P(y=k|x) = P(y=l|x),
which simplifies to (β_k − β_l)ᵀx = const — a **linear hyperplane**.
The full decision space is partitioned into K convex regions.

In [ ]:
le = LabelEncoder().fit(iris['species'])
y_iris = le.transform(iris['species'])
X_iris = iris[['petal_length', 'petal_width']].to_numpy()

m_mn = LogisticRegression(max_iter=2000).fit(X_iris, y_iris)
print('Coefficient matrix shape:', m_mn.coef_.shape, '  (one row per class)')
print(f'Training accuracy = {m_mn.score(X_iris, y_iris):.4f}')
print('\nCoefficient matrix (row = class, col = feature):')
print(pd.DataFrame(m_mn.coef_, index=le.classes_, columns=['petal_length', 'petal_width']).round(4))

In [ ]:
x_min, x_max = X_iris[:, 0].min() - 0.5, X_iris[:, 0].max() + 0.5
y_min, y_max = X_iris[:, 1].min() - 0.5, X_iris[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
Z = m_mn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx, yy, Z, alpha=0.25, levels=[-0.5, 0.5, 1.5, 2.5], cmap='Set1')
for cls in range(3):
    ax.scatter(X_iris[y_iris==cls, 0], X_iris[y_iris==cls, 1],
               label=le.classes_[cls], s=25, edgecolors='k', linewidths=0.5)
ax.set_xlabel('petal_length'); ax.set_ylabel('petal_width')
ax.set_title('Multinomial LR on Iris — linear decision regions')
ax.legend(); plt.show()

In [ ]:
# Probabilities sum to 1 — a valid distribution
sample_proba = m_mn.predict_proba(X_iris[:5])
print('Probabilities for first 5 flowers:')
print(pd.DataFrame(sample_proba, columns=le.classes_).round(4))
print('\nRow sums:', sample_proba.sum(axis=1).round(6))  # all 1.0

## 6. ROC Curves, AUC, and Threshold Tuning

### Confusion matrix vocabulary

|              | Predicted: No | Predicted: Yes |
|--------------|--------------|----------------|
| **Actual: No**  | True Negative (TN) | False Positive (FP) |
| **Actual: Yes** | False Negative (FN) | True Positive (TP) |

- **FPR** = FP / (FP + TN) — fraction of negatives incorrectly flagged (Type I error)
- **FNR** = FN / (FN + TP) — fraction of positives missed (Type II error)
- **TPR = Recall** = TP / (TP + FN) = 1 − FNR

The right tradeoff depends on **cost ratio**: how expensive is a false negative vs a false positive?

### ROC curve

Sweep the classification threshold from 1 → 0. At each threshold compute (FPR, TPR).
Plot TPR vs FPR: the **Receiver Operating Characteristic** curve.

**AUC** = area under ROC = P(score(positive) > score(negative)). Model selection metric; independent of threshold.

In [ ]:
X_def = d[['balance', 'income']].to_numpy()
Xtr_d, Xte_d, ytr_d, yte_d = train_test_split(X_def, y_bin,
                                                test_size=0.3, random_state=0, stratify=y_bin)
m_roc = LogisticRegression(max_iter=2000).fit(Xtr_d, ytr_d)
proba_d = m_roc.predict_proba(Xte_d)[:, 1]
print(f'First 5 probabilities: {proba_d[:5].round(4)}')
print(f'First 5 predictions (t=0.5): {(proba_d[:5] >= 0.5).astype(int)}')

In [ ]:
# Confusion matrix at default threshold 0.5
pred_default = (proba_d >= 0.5).astype(int)
cm = confusion_matrix(yte_d, pred_default)
tn, fp, fn, tp = cm.ravel()
print('Confusion matrix at threshold = 0.5:')
print(pd.DataFrame(cm, index=['Actual: No', 'Actual: Yes'],
                   columns=['Pred: No', 'Pred: Yes']))
print(f'\nFPR = {fp/(fp+tn):.4f}  (false alarm rate)')
print(f'FNR = {fn/(fn+tp):.4f}  (missed defaulters — costly!)')
print(f'TPR = {tp/(tp+fn):.4f}  (recall)')

In [ ]:
fpr, tpr, thr = roc_curve(yte_d, proba_d)
auc_score = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot(fpr, tpr, linewidth=2, label=f'LR (AUC = {auc_score:.4f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random (AUC = 0.5)')
ax.set_xlabel('FPR (False Positive Rate)'); ax.set_ylabel('TPR (True Positive Rate)')
ax.set_title('ROC Curve — Default dataset')
ax.legend(); plt.show()

In [ ]:
# Threshold sensitivity table (recreates ISLR Table 4.4)
print('Threshold sensitivity (Default dataset):')
print(f'{"Threshold":>12} {"FPR":>8} {"FNR":>8} {"Accuracy":>12}')
for t in [0.10, 0.20, 0.50]:
    pred_t = (proba_d >= t).astype(int)
    cm_t = confusion_matrix(yte_d, pred_t).ravel()
    tn_t, fp_t, fn_t, tp_t = cm_t
    fpr_t = fp_t / (fp_t + tn_t)
    fnr_t = fn_t / (fn_t + tp_t)
    acc_t = (tn_t + tp_t) / len(yte_d)
    print(f'{t:>12.2f} {fpr_t:>8.3f} {fnr_t:>8.3f} {acc_t:>12.3f}')

In [ ]:
# Sweep threshold to maximize F1
ts = np.linspace(0.02, 0.6, 80)
f1s = [f1_score(yte_d, (proba_d >= t).astype(int), zero_division=0) for t in ts]
best_t = ts[int(np.argmax(f1s))]

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(ts, f1s, linewidth=2)
ax.axvline(best_t, color='C3', linestyle='--', label=f'Best threshold = {best_t:.2f}')
ax.axvline(0.5, color='gray', linestyle=':', label='Default threshold = 0.50')
ax.set_xlabel('Threshold'); ax.set_ylabel('F1 score')
ax.set_title('F1 vs threshold (Default dataset)')
ax.legend(); plt.show()

print(f'F1 at threshold = 0.50: {f1_score(yte_d, (proba_d>=0.5).astype(int), zero_division=0):.4f}')
print(f'F1 at threshold = {best_t:.2f}: {max(f1s):.4f}')

## 7. L1 and L2 Regularization

The regularized objective:
$$\text{maximize} \quad \ell(\beta) - \lambda \cdot \text{penalty}(\beta)$$

- **L2 (Ridge)**: penalty = ½ Σⱼ β²ⱼ — shrinks all coefficients toward zero smoothly
- **L1 (Lasso)**: penalty = Σⱼ |βⱼ| — exactly zeros some coefficients (sparse)

sklearn convention: **C = 1/λ**. Small C = strong regularization. Default: `C=1.0, penalty='l2'`.

Always **standardize features** before regularizing — otherwise large-scale features are penalized less.

In [ ]:
X4 = iris[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']].to_numpy()
feat_names = ['sepal_len', 'sepal_wid', 'petal_len', 'petal_wid']
scaler = StandardScaler().fit(X4)
Xs4 = scaler.transform(X4)
Cs = np.logspace(-3, 2, 40)

paths_l2 = np.empty((len(Cs), 3, 4))
paths_l1 = np.empty((len(Cs), 3, 4))
for i, C in enumerate(Cs):
    paths_l2[i] = LogisticRegression(C=C, max_iter=3000).fit(Xs4, y_iris).coef_
    paths_l1[i] = LogisticRegression(C=C, penalty='l1', solver='saga',
                                      max_iter=3000).fit(Xs4, y_iris).coef_

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
colors = ['C0', 'C1', 'C2', 'C3']
for ax, paths, title in zip(axes, [paths_l2, paths_l1], ['L2 (Ridge)', 'L1 (Lasso)']):
    for j, (name, col) in enumerate(zip(feat_names, colors)):
        ax.plot(Cs, paths[:, 0, j], color=col, label=name, linewidth=1.5)
    ax.set_xscale('log'); ax.set_xlabel('C (= 1/λ)')
    ax.set_ylabel('Coefficient (class: setosa)')
    ax.set_title(f'{title} coefficient paths')
    ax.legend(fontsize=9)
    ax.axhline(0, color='k', linewidth=0.7, alpha=0.4)
plt.tight_layout(); plt.show()
print('L1: notice coefficients hit exactly zero (sparse) for small C.')
print('L2: coefficients shrink smoothly but never reach zero.')

In [ ]:
# Which features survive L1 with C=0.05?
m_l1 = LogisticRegression(C=0.05, penalty='l1', solver='saga', max_iter=5000).fit(Xs4, y_iris)
print('Survivors under L1 (C=0.05):')
for cls, row in zip(le.classes_, m_l1.coef_):
    survivors = [n for n, c in zip(feat_names, row) if abs(c) > 1e-6]
    print(f'  {cls:12s}: {survivors}')

## 8. When Linear Fails — Motivating Non-Linear Classifiers

A logistic regression model has a **linear decision boundary** in the original feature space.
No amount of training data or regularization tuning can make it curved.

### The make_moons problem

In [ ]:
Xm, ym = make_moons(n_samples=400, noise=0.25, random_state=0)
m_moons = LogisticRegression(max_iter=2000).fit(Xm, ym)
print(f'LR on moons — training accuracy = {m_moons.score(Xm, ym):.4f}')
print('A linear boundary cannot capture the curved shape. Performance is capped ~85%.')

xx, yy = np.meshgrid(np.linspace(Xm[:,0].min()-0.5, Xm[:,0].max()+0.5, 300),
                      np.linspace(Xm[:,1].min()-0.5, Xm[:,1].max()+0.5, 300))
Z = m_moons.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
fig, ax = plt.subplots(figsize=(6, 5))
ax.contourf(xx, yy, Z, alpha=0.25, cmap='RdBu')
ax.scatter(Xm[ym==0, 0], Xm[ym==0, 1], s=12, label='Class 0')
ax.scatter(Xm[ym==1, 0], Xm[ym==1, 1], s=12, color='C3', label='Class 1')
ax.set_title('Moons dataset: a line cannot separate two moons')
ax.legend(); plt.show()

In [ ]:
# Exit 1: polynomial features
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

pipe_poly = Pipeline([
    ('poly', PolynomialFeatures(degree=3, include_bias=False)),
    ('lr', LogisticRegression(max_iter=2000))
])
pipe_poly.fit(Xm, ym)
print(f'Polynomial features (degree=3) + LR: accuracy = {pipe_poly.score(Xm, ym):.4f}')
print('Still LR, but the feature expansion allows a curved effective boundary.')
print('Courses 7-9: decision trees, ensembles, and SVMs will handle this natively.')

## Recap

1. **Linear regression can't model probabilities.** Always use logistic for binary Y.
2. **Log-odds = β₀ + β₁x.** eᵝ is the odds ratio per unit of x.
3. **MLE is concave** — unique global optimum, fast to solve.
4. **Confounding reverses signs** — marginal and conditional associations can completely disagree.
5. **Case-control sampling?** Only the intercept is biased; correct it with the log-prevalence ratio.
6. **Softmax** generalizes sigmoid to K classes. K−1 identifiable weight vectors.
7. **Tune the threshold.** 0.5 is almost never right on imbalanced data. Use AUC to compare models.
8. **C = 1/λ.** L1 zeros coefficients. L2 shrinks them. Always standardize first.

---

# Exercises

## Exercise 1

**Task 1.** Load `titanic`, drop rows missing `age`, one-hot encode `sex` and `embarked`.
Fit binary logistic regression predicting `survived` from `pclass, sex, age, sibsp, parch, fare, embarked`.
Report held-out accuracy and AUC.

In [ ]:
# your code here

### Exercise 1 — Solution

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

df = load_dataset('titanic').dropna(subset=['age']).reset_index(drop=True)
cols = ['survived','pclass','sex','age','sibsp','parch','fare','embarked']
df = pd.get_dummies(df[cols].fillna(df.median(numeric_only=True)),
                    columns=['sex','embarked'], drop_first=True, dtype=float)
y = df['survived']; X = df.drop(columns=['survived'])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
m = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
print(f'accuracy = {m.score(Xte, yte):.4f}')
print(f'AUC      = {roc_auc_score(yte, m.predict_proba(Xte)[:,1]):.4f}')

## Exercise 2

**Task 2.** Tune the decision threshold for maximum F1 on the held-out Titanic set.

In [ ]:
# your code here

### Exercise 2 — Solution

In [ ]:
import numpy as np
from sklearn.metrics import f1_score
p = m.predict_proba(Xte)[:, 1]
ts = np.linspace(0.1, 0.9, 81)
f1s = [f1_score(yte, (p >= t).astype(int)) for t in ts]
best_t = ts[int(np.argmax(f1s))]
print(f'F1@0.5     = {f1_score(yte, (p>=0.5).astype(int)):.4f}')
print(f'F1@{best_t:.2f}  = {max(f1s):.4f}')

## Exercise 3

**Task 3.** Add an interaction `sex_male × pclass` to the Titanic design matrix.
Does AUC improve?

In [ ]:
# your code here

### Exercise 3 — Solution

In [ ]:
X2 = X.copy()
X2['sex_x_pclass'] = X['sex_male'] * X['pclass']
Xtr2, Xte2, _, _ = train_test_split(X2, y, test_size=0.3, random_state=0, stratify=y)
m2 = LogisticRegression(max_iter=2000).fit(Xtr2, ytr)
print(f'Baseline AUC   = {roc_auc_score(yte, m.predict_proba(Xte)[:,1]):.4f}')
print(f'With interaction = {roc_auc_score(yte, m2.predict_proba(Xte2)[:,1]):.4f}')

## Exercise 4

**Task 4.** Fit plain multinomial logistic regression on `penguins`
to predict `species` from all four numeric measurements. Report 5-fold CV accuracy. Drop NaN first.

In [ ]:
# your code here

### Exercise 4 — Solution

In [ ]:
from shared.data_utils import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
df = load_dataset('penguins').dropna()
X_p = df[['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']]
y_p = df['species']
pipe = Pipeline([('s', StandardScaler()), ('lr', LogisticRegression(max_iter=2000))])
print(f'CV accuracy = {cross_val_score(pipe, X_p, y_p, cv=5).mean():.4f}')

## Exercise 5

**Task 5.** Fit L1-penalized logistic regression on penguins with `C = 0.05`.
Which features survive (are non-zero) for each class?

In [ ]:
# your code here

### Exercise 5 — Solution

In [ ]:
import numpy as np
Xs_p = StandardScaler().fit_transform(X_p)
m_l1 = LogisticRegression(C=0.05, penalty='l1', solver='saga', max_iter=5000).fit(Xs_p, y_p)
for cls, row in zip(m_l1.classes_, m_l1.coef_):
    keep = [n for n, c in zip(X_p.columns, row) if abs(c) > 1e-6]
    print(f'{cls:12s}: survivors = {keep}')

## Exercise 6

**Task 6.** Compare validation accuracy for `C ∈ [0.001, 0.01, 0.1, 1, 10]` under L1 on penguins.
Plot accuracy vs C on a log axis.

In [ ]:
# your code here

### Exercise 6 — Solution

In [ ]:
import matplotlib.pyplot as plt
Cs_grid = [0.001, 0.01, 0.1, 1, 10]
accs = []
for C in Cs_grid:
    pipe_l1 = Pipeline([('s', StandardScaler()),
                        ('lr', LogisticRegression(C=C, penalty='l1', solver='saga', max_iter=5000))])
    accs.append(cross_val_score(pipe_l1, X_p, y_p, cv=5).mean())
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogx(Cs_grid, accs, marker='o', linewidth=2)
ax.set_xlabel('C (= 1/λ)'); ax.set_ylabel('5-fold CV accuracy')
ax.set_title('L1 regularization path — penguins')
plt.show()
for C, a in zip(Cs_grid, accs): print(f'C={C:7g}  acc = {a:.4f}')